# Data Sprint 2.0 NLP: reproduce candidate NEW-G (v007 models, geometric blend)
**Model:** HingRoBERTa-mixed (fine-tuned, 278,086,712 params) + character TF-IDF / LinearSVC (9,291,576 params), combined by a **geometric mean** (average of log-probabilities, 50/50) instead of v007's arithmetic mean, then a class-55 fallback rule with per-class biases. **Total 287,378,288 parameters** (limit 300,000,000). Everything else is identical to v007.

**Run:** Accelerator = GPU (T4 or P100), Internet = On (to download the pretrained HingRoBERTa from Hugging Face), competition data attached. *Run All* takes about 25 minutes: TF-IDF 5-fold + full fit on CPU (~5 min), HingRoBERTa full-data fine-tuning on GPU (~16 min), blending (seconds).

**What it uses:** only the competition's `train.csv` / `test.csv`, the open-source pretrained checkpoint `l3cube-pune/hing-roberta-mixed` (CC-BY-4.0), and rule-based augmentation (`augment.py`). No external datasets, no commercial LLM APIs, no hand labelling. The decision rule below was tuned on 5-fold out-of-fold predictions of `train.csv` (section 7 re-derives it; optional, ~65 extra GPU minutes).

Sections: 1 config · 2 data · 3 source files · 4 frozen rule · 5 TF-IDF · 6 HingRoBERTa · 7 optional rule re-tuning · 8 blend + submission · 9 parameter check · 10 agreement with our submitted file.

In [ ]:
# 1) config
SMOKE = False            # True = tiny CPU run (300 rows, 1 epoch) just to test the plumbing
RECOMPUTE_RULE = False   # True = re-derive weights/tau/bias from 5-fold OOF (adds ~65 GPU minutes)
SEED = 42
REPO = '/kaggle/working/repo' if __import__('os').path.isdir('/kaggle/working') else 'repo'


In [ ]:
# 2) data: find the competition CSVs and place them where the scripts expect them
import os, glob, shutil, subprocess, json
import pandas as pd
os.makedirs(f'{REPO}/data', exist_ok=True); os.makedirs(f'{REPO}/src', exist_ok=True)
os.makedirs(f'{REPO}/kaggle', exist_ok=True); os.makedirs(f'{REPO}/submissions', exist_ok=True)
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True)) or sorted(glob.glob(f'**/data/{name}', recursive=True))
    assert hits, f'{name} not found: attach the competition data'
    return hits[0]
for n in ['train.csv', 'test.csv']:
    if not os.path.exists(f'{REPO}/data/{n}'):
        shutil.copy(find(n), f'{REPO}/data/{n}')
tr, te = pd.read_csv(f'{REPO}/data/train.csv'), pd.read_csv(f'{REPO}/data/test.csv')
print('train', tr.shape, '| test', te.shape, '| classes', tr.target.nunique())
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 3) Source files
Exact copies of the files in our repository (`src/`, `kaggle/train.py`). `normalize.py`: Devanagari→Latin transliteration and Hinglish phonetic key. `augment.py`: rule-based test-style augmentation (romanisation, fillers, digits, ASR noise). `baseline.py`: TF-IDF + LinearSVC. `train.py`: transformer fine-tuning. `postprocess.py` / `testlike.py`: OOF blending, rule tuning and the test-like validation subset (used only in optional section 7).

In [ ]:
%%writefile {REPO}/src/normalize.py
"""
Text normalization for Hindi / Hinglish voice queries.

transliterate_indic() is reused from our Amazon ML Challenge 2026 entity-resolution pipeline
(rule-based Devanagari -> Latin). phonetic_key() folds common Hinglish spelling / ASR variants
("chahoonga" ~ "chahunga", "kya" ~ "kia") so Romanized and Devanagari queries meet in one space.
"""
import re
import unicodedata

# ---------------------------------------------------------------------------
# Indic -> Latin transliteration
# ---------------------------------------------------------------------------
INDIC_BLOCKS = [0x0900, 0x0980, 0x0A00, 0x0A80, 0x0B00, 0x0B80, 0x0C00, 0x0C80, 0x0D00]

_CONS = {  # offset -> latin consonant (inherent vowel added separately)
    0x15: "k", 0x16: "kh", 0x17: "g", 0x18: "gh", 0x19: "n", 0x1A: "ch", 0x1B: "chh",
    0x1C: "j", 0x1D: "jh", 0x1E: "n", 0x1F: "t", 0x20: "th", 0x21: "d", 0x22: "dh",
    0x23: "n", 0x24: "t", 0x25: "th", 0x26: "d", 0x27: "dh", 0x28: "n", 0x29: "n",
    0x2A: "p", 0x2B: "ph", 0x2C: "b", 0x2D: "bh", 0x2E: "m", 0x2F: "y", 0x30: "r",
    0x31: "r", 0x32: "l", 0x33: "l", 0x34: "zh", 0x35: "v", 0x36: "sh", 0x37: "sh",
    0x38: "s", 0x39: "h",
    0x58: "q", 0x59: "kh", 0x5A: "g", 0x5B: "z", 0x5C: "r", 0x5D: "rh", 0x5E: "f", 0x5F: "y",
}
_VOWEL = {  # independent vowels
    0x05: "a", 0x06: "a", 0x07: "i", 0x08: "i", 0x09: "u", 0x0A: "u", 0x0B: "ri", 0x0C: "l",
    0x0D: "e", 0x0E: "e", 0x0F: "e", 0x10: "ai", 0x11: "o", 0x12: "o", 0x13: "o", 0x14: "au",
    0x60: "ri", 0x61: "l", 0x50: "om",
}
_SIGN = {  # dependent vowel signs (replace inherent 'a')
    0x3E: "a", 0x3F: "i", 0x40: "i", 0x41: "u", 0x42: "u", 0x43: "ri", 0x44: "ri",
    0x45: "e", 0x46: "e", 0x47: "e", 0x48: "ai", 0x49: "o", 0x4A: "o", 0x4B: "o",
    0x4C: "au", 0x62: "l", 0x63: "l",
}
_NASAL = {0x01: "n", 0x02: "n", 0x03: "h", 0x70: "n"}  # candrabindu, anusvara, visarga, gurmukhi tippi
_VIRAMA = 0x4D
_CHILLU = ["n", "n", "r", "l", "l", "k"]
_INDIC_RE = re.compile("[ऀ-ൿ]")


def _indic_offset(ch):
    cp = ord(ch)
    if 0x0900 <= cp <= 0x0D7F:
        return cp & 0x7F  # every block is 0x80 wide
    return None


def transliterate_indic(text: str) -> str:
    if not _INDIC_RE.search(text):
        return text
    out = []
    pending = False  # a consonant waiting for its vowel
    word_len = 0  # Indic code points seen in the current word
    for ch in text:
        off = _indic_offset(ch)
        if off is None:
            if pending and word_len <= 1:
                out.append("a")  # word-final schwa is dropped except in one-letter words
            pending = False
            word_len = 0
            out.append(ch)
            continue
        word_len += 1
        if off in _CONS:
            if pending:
                out.append("a")
            out.append(_CONS[off])
            pending = True
        elif off in _SIGN:
            out.append(_SIGN[off])
            pending = False
        elif off == _VIRAMA:
            pending = False
        elif 0x0D7A <= ord(ch) <= 0x0D7F:  # Malayalam chillu letters: consonant without vowel
            if pending:
                out.append("a")
            out.append(_CHILLU[ord(ch) - 0x0D7A])
            pending = False
        elif off in _VOWEL:
            if pending:
                out.append("a")
                pending = False
            out.append(_VOWEL[off])
        elif off in _NASAL:
            if pending:
                out.append("a")
                pending = False
            out.append(_NASAL[off])
        elif 0x66 <= off <= 0x6F:  # native digits
            if pending:
                out.append("a")
                pending = False
            out.append(str(off - 0x66))
        else:  # nukta, danda, length marks, etc.
            if off in (0x64, 0x65):
                if pending:
                    out.append("a")
                    pending = False
                out.append(" ")
    if pending and word_len <= 1:
        out.append("a")
    return "".join(out)


_NON_ALNUM = re.compile(r"[^a-z0-9]+")
_PHON = [
    (re.compile(r"(.)\1+"), r"\1"),        # collapse doubled letters: baje/bajje, karna/karrna
    (re.compile(r"ee|ii|ie"), "i"),
    (re.compile(r"oo|uu|ou"), "u"),
    (re.compile(r"aa"), "a"),
    (re.compile(r"w"), "v"),
    (re.compile(r"ph"), "f"),
    (re.compile(r"z"), "j"),
    (re.compile(r"q"), "k"),
    (re.compile(r"c(?!h)"), "k"),
    (re.compile(r"(?<=[a-z])h(?=[^aeiou]|$)"), ""),  # drop non-initial aspiration h: bhai keeps, kah->ka
    (re.compile(r"y(?=[aeiou])"), "i"),     # kya ~ kia
]


def clean(text: str) -> str:
    """Lowercase, NFKC, Devanagari -> Latin, strip punctuation."""
    t = unicodedata.normalize("NFKC", str(text)).lower()
    t = transliterate_indic(t)
    t = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    return _NON_ALNUM.sub(" ", t).strip()


def phonetic_key(text: str) -> str:
    """clean() plus Hinglish phonetic folding, token by token."""
    out = []
    for tok in clean(text).split():
        if not tok.isdigit():
            for pat, rep in _PHON:
                tok = pat.sub(rep, tok)
        out.append(tok)
    return " ".join(out)


if __name__ == "__main__":
    for s in ["मैं चाहता हूं कि इस घटना को मेरे कैलेंडर से हटा दिया जाए",
              "main chahta hoon ki is ghatna ko mere calendar se hata diya jaye",
              "kia ap imel kholenge", "क्या आप ईमेल खोलेंगे"]:
        print(phonetic_key(s))


In [ ]:
%%writefile {REPO}/src/augment.py
"""
Train-time augmentations that move the training distribution towards the test distribution.

Measured shift (analysis/eda.py):
  * script flip: train 61% Devanagari / 33% Roman, test 33% Devanagari / 64% Roman
  * colloquial register in test only (bhai / please / na / jaldi / hey ~3% of rows each, ~0 in train)
  * digits appear in test text (1.3% of rows), none in train
  * ASR-style spellings ("mjhe", "volyoom")

All functions are deterministic given an `rng`, rule-based, and use no external data or LLMs.
The official task description itself uses this register: "bhai kal subah 7 baje ka alarm laga de".

Two modes (environment variable AUG_MODE, read at call time):
  * "v007" (default): exactly the augmentation behind the submitted v007, including the random-number sequence.
  * "mined": filler words and respellings MINED from the competition text (train + unlabelled test) instead of the
    fixed PREFIX/SUFFIX lists, plus a respelling step (train transliteration -> spelling seen in test, e.g.
    kofee -> coffee). Nothing hand-written: `mine_fillers()` / `mine_spellings()` derive the lists from the CSVs.
"""
import collections
import functools
import os
import random
import re
from pathlib import Path

from normalize import _CONS  # Devanagari consonant table reused from the Amazon ML Challenge pipeline

DEV = re.compile("[ऀ-ॿ]")
_NUKTA = {"ph": "f", "j": "z", "d": "r", "dh": "rh", "k": "k", "kh": "kh", "g": "g"}
_SIGN = {0x3E: ("aa", "a"), 0x40: ("ee", "i"), 0x42: ("oo", "u"), 0x3F: ("i", "i"), 0x41: ("u", "u"),
         0x47: ("e", "e"), 0x48: ("ai", "ai"), 0x4B: ("o", "o"), 0x4C: ("au", "au"), 0x45: ("e", "e"),
         0x49: ("o", "o"), 0x43: ("ri", "ri"), 0x44: ("ri", "ri")}
_VOW = {0x05: ("a", "a"), 0x06: ("aa", "a"), 0x07: ("i", "i"), 0x08: ("ee", "i"), 0x09: ("u", "u"),
        0x0A: ("oo", "u"), 0x0B: ("ri", "ri"), 0x0F: ("e", "e"), 0x10: ("ai", "ai"), 0x13: ("o", "o"),
        0x14: ("au", "au"), 0x11: ("o", "o"), 0x0D: ("e", "e")}
_DIGIT_DEV = {chr(0x0966 + i): str(i) for i in range(10)}


def _romanize_word(word, long_double):
    """Devanagari word -> Roman in the dataset's style (aa/ee/oo for long vowels, Hindi schwa deletion)."""
    syl = []  # [consonant, vowel, inherent?]
    for ch in word:
        cp = ord(ch)
        if not 0x0900 <= cp <= 0x097F:
            syl.append([ch, "", False])
            continue
        off = cp - 0x0900
        if off in _CONS:
            syl.append([_CONS[off], "a", True])
        elif off == 0x3C and syl:  # nukta
            syl[-1][0] = _NUKTA.get(syl[-1][0], syl[-1][0])
        elif off in _SIGN and syl:
            syl[-1][1], syl[-1][2] = _SIGN[off][0 if long_double else 1], False
        elif off == 0x4D and syl:  # virama
            syl[-1][1], syl[-1][2] = "", False
        elif off in _VOW:
            syl.append(["", _VOW[off][0 if long_double else 1], False])
        elif off in (0x01, 0x02):  # candrabindu / anusvara
            if syl:
                syl[-1][1] += "n"
                syl[-1][2] = False
        elif off == 0x03 and syl:
            syl[-1][1] += "h"
        elif chr(cp) in _DIGIT_DEV:
            syl.append([_DIGIT_DEV[chr(cp)], "", False])
    # schwa deletion, right to left: drop final schwa; drop medial schwa in V C(a) C V
    n = len(syl)
    for i in range(n - 1, -1, -1):
        if not syl[i][2]:
            continue
        if i == n - 1 and n > 1:
            syl[i][1] = ""
        elif 0 < i < n - 1 and syl[i - 1][1] and syl[i + 1][0] and syl[i + 1][1]:
            syl[i][1] = ""
    return "".join(c + v for c, v, _ in syl)


def romanize(text, rng=None, p_long=0.75):
    """Romanize every Devanagari word; long vowels written doubled with prob p_long (per word)."""
    rng = rng or random
    return " ".join(_romanize_word(w, rng.random() < p_long) if DEV.search(w) else w for w in text.split())


# --- colloquial register --------------------------------------------------------------------
PREFIX = ["bhai", "yaar", "hey", "please", "plz", "zara", "ok", "suno", "achha"]
SUFFIX = ["na", "yaar", "bhai", "please", "jaldi", "plz", "jaldi se", "abhi"]

# --- mined word lists (AUG_MODE=mined) -------------------------------------------------------
DATA = Path(__file__).resolve().parents[1] / "data"
_ALPHA = re.compile("^[a-z]+$")


def mode():
    return os.environ.get("AUG_MODE", "v007")


@functools.lru_cache(maxsize=1)
def _texts():
    import pandas as pd
    try:
        tr = pd.read_csv(DATA / "train.csv").query_text.str.lower().tolist()
        te = pd.read_csv(DATA / "test.csv").query_text.str.lower().tolist()
    except FileNotFoundError:
        return [], []
    return tr, te


@functools.lru_cache(maxsize=1)
def mine_fillers(min_count=12, max_train_ratio=0.2, min_diversity=0.5, min_edge_share=0.6):
    """(prefixes, suffixes) mined from the text: edge words/bigrams frequent at the start/end of test queries, rare
    there in train, attached to many different neighbouring words, and mostly occurring at a query edge."""
    tr, te = _texts()
    if not te:
        return (), ()
    scale = len(te) / max(len(tr), 1)
    tok_te = collections.Counter(w for s in te for w in s.split())
    edge_te = collections.Counter(x for s in te if len(s.split()) >= 3 for x in {s.split()[0], s.split()[-1]})
    out = []
    for side in ("pre", "suf1", "suf2"):
        cnt, ctx, cnt_tr = collections.Counter(), collections.defaultdict(set), collections.Counter()
        for texts, c in ((te, cnt), (tr, cnt_tr)):
            for s in texts:
                w = s.split()
                if len(w) < 3:
                    continue
                k, nb = {"pre": (w[0], w[1]), "suf1": (w[-1], w[-2]), "suf2": (" ".join(w[-2:]), w[-3])}[side]
                if all(_ALPHA.match(x) for x in k.split()):
                    c[k] += 1
                    if texts is te:
                        ctx[k].add(nb)
        keep = [k for k, v in cnt.items()
                if v >= min_count and cnt_tr[k] * scale < max_train_ratio * v and len(ctx[k]) / v >= min_diversity
                and (" " in k or edge_te[k] / tok_te[k] >= min_edge_share)]
        out.append(sorted(keep))
    return tuple(out[0]), tuple(out[1] + out[2])


@functools.lru_cache(maxsize=1)
def mine_spellings(min_len=4):
    """{train spelling: test spelling} for Roman words whose phonetic key matches an unseen test spelling."""
    from normalize import clean, phonetic_key
    tr, te = _texts()
    if not te:
        return {}
    v_tr = collections.Counter(w for s in tr for w in clean(s).split())
    v_te = collections.Counter(w for s in te for w in clean(s).split())
    by_key = collections.defaultdict(list)
    for w, c in v_tr.items():
        if len(w) >= min_len and _ALPHA.match(w):
            by_key[phonetic_key(w)].append((c, w))
    out = {}
    for w, c in v_te.items():
        if len(w) >= min_len and w not in v_tr and _ALPHA.match(w):
            k = phonetic_key(w)
            if k in by_key:
                out[max(by_key[k])[1]] = w
    return out


def respell(text, rng, p=0.7):
    """Swap train transliterations for the spelling seen in test (kofee -> coffee) with prob p per word."""
    m = mine_spellings()
    return " ".join(m[w] if w in m and rng.random() < p else w for w in text.split())


def colloquial(text, rng):
    pre, suf = (PREFIX, SUFFIX) if mode() != "mined" else mine_fillers()
    if rng.random() < 0.5 and pre:
        text = f"{rng.choice(pre)} {text}"
    if rng.random() < 0.6 and suf:
        text = f"{text} {rng.choice(suf)}"
    return text


# --- number words -> digits -----------------------------------------------------------------
NUM = {"ek": 1, "do": 2, "teen": 3, "tin": 3, "char": 4, "chaar": 4, "paanch": 5, "panch": 5, "chhah": 6,
       "chhe": 6, "chah": 6, "saat": 7, "sat": 7, "aath": 8, "ath": 8, "nau": 9, "das": 10, "gyaarah": 11,
       "gyarah": 11, "baarah": 12, "barah": 12, "pandrah": 15, "bees": 20, "tees": 30, "pachaas": 50, "sau": 100,
       "एक": 1, "दो": 2, "तीन": 3, "चार": 4, "पांच": 5, "पाँच": 5, "छह": 6, "छः": 6, "सात": 7, "आठ": 8, "नौ": 9,
       "दस": 10, "ग्यारह": 11, "बारह": 12, "पंद्रह": 15, "बीस": 20, "तीस": 30, "पचास": 50, "सौ": 100}
_TIME_CTX = {"baje", "bje", "बजे", "minute", "minat", "मिनट", "ghante", "घंटे", "din", "दिन", "pm", "am"}


def digits(text, rng):
    """Replace number words by digits when followed by a time/quantity word ('saat baje' -> '7 baje')."""
    toks = text.split()
    for i, t in enumerate(toks[:-1]):
        if t in NUM and (toks[i + 1] in _TIME_CTX or rng.random() < 0.3) and t not in ("do", "दो") or (
                t in ("do", "दो") and toks[i + 1] in _TIME_CTX):
            toks[i] = str(NUM[t])
    return " ".join(toks)


# --- ASR / typing noise ---------------------------------------------------------------------
def asr_noise(text, rng, p=0.12):
    out = []
    for w in text.split():
        r = rng.random()
        if len(w) > 3 and r < p and not DEV.search(w):  # drop an inner vowel: mujhe -> mjhe
            idx = [i for i, c in enumerate(w[1:-1], 1) if c in "aeiou"]
            if idx:
                i = rng.choice(idx)
                w = w[:i] + w[i + 1:]
        elif len(w) > 3 and r < 2 * p and not DEV.search(w):  # vowel length swap: ee<->i, oo<->u, aa<->a
            w = rng.choice([w.replace("ee", "i"), w.replace("i", "ee", 1), w.replace("aa", "a"),
                            w.replace("oo", "u"), w.replace("u", "oo", 1)])
        elif len(w) > 3 and r < 2 * p and DEV.search(w):  # drop a matra in Devanagari
            idx = [i for i, c in enumerate(w) if 0x093E <= ord(c) <= 0x094C]
            if idx:
                i = rng.choice(idx)
                w = w[:i] + w[i + 1:]
        elif len(w) > 5 and r < 2.5 * p:  # split word (ASR segmentation)
            i = rng.randint(2, len(w) - 2)
            w = w[:i] + " " + w[i:]
        out.append(w)
    return " ".join(out)


def augment(text, rng, p_roman=0.7, p_coll=0.35, p_digit=0.5, p_noise=0.5, p_respell=0.5):
    """One random view of `text` drawn to look like the test distribution."""
    if DEV.search(text) and rng.random() < p_roman:
        text = romanize(text, rng)
    if mode() == "mined" and rng.random() < p_respell:
        text = respell(text, rng)
    if rng.random() < p_digit:
        text = digits(text, rng)
    if rng.random() < p_coll:
        text = colloquial(text, rng)
    if rng.random() < p_noise:
        text = asr_noise(text, rng)
    return text


if __name__ == "__main__":
    rng = random.Random(0)
    for s in ["अगली सुबह सात बजे के लिए अलार्म", "मुझे गर्म कॉफी चाहिए", "दिल्ली के आसपास सर्वोत्तम रेस्टोरेंट कौन सा है",
              "मुझे इस कलाकार की आवाज़ से नफ़रत है", "kripaya rasoeeghar ki battiyaan band kar den"]:
        print(romanize(s, random.Random(1)), " | ", augment(s, rng))


In [ ]:
%%writefile {REPO}/src/baseline.py
"""
CPU baseline: TF-IDF (char + word n-grams on raw text and on the phonetic key) + LinearSVC.

    python src/baseline.py            # 5-fold CV macro-F1 + writes submissions/baseline_tfidf.csv
    python src/baseline.py --aug      # + test-style augmentation (romanize / colloquial / digits / ASR noise)
    python src/baseline.py --aug --save_run runs/tfidf_aug   # also write fold outputs blendable by postprocess.py
    python src/baseline.py --aug --pseudo --save_run runs/tfidf_aug_ps   # + pseudo-labelled test rows
      (fold k adds data/pseudo_f{k}_t92.csv, teacher = fold-k models only, which never saw fold k; the full fit adds pseudo_all_t92.csv)

--save_run writes <dir>/fold{k}/{val_idx,val_logits,test_logits}.npy + metrics.json in the same format and with
the same StratifiedKFold(5, shuffle, 42) splits as kaggle/train.py. SVM margins are scaled by one temperature
fitted on OOF log-loss so that softmax gives usable probabilities for blending.
"""
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

sys.path.insert(0, str(Path(__file__).parent))
from augment import augment  # noqa: E402
from normalize import clean, phonetic_key  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
SEED = 42


def vectorizers():
    return [
        ("raw_char", lambda d: d.raw, TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ("key_char", lambda d: d.key, TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ("key_word", lambda d: d.key, TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ]


def featurize(fit_df, other_dfs):
    mats_fit, mats_other = [], [[] for _ in other_dfs]
    for _, col, vec in vectorizers():
        mats_fit.append(vec.fit_transform(col(fit_df)))
        for i, d in enumerate(other_dfs):
            mats_other[i].append(vec.transform(col(d)))
    return hstack(mats_fit).tocsr(), [hstack(m).tocsr() for m in mats_other]


def prep(df):
    df = df.copy()
    df["raw"] = df.query_text.map(clean)
    df["key"] = df.query_text.map(phonetic_key)
    df["is_dev"] = df.query_text.str.contains("[ऀ-ॿ]")
    return df


def with_aug(df, seed, copies=2):
    """Training rows plus `copies` augmented views of each (labels kept)."""
    rng = random.Random(seed)
    views = [df[["query_text", "target"]]]
    for _ in range(copies):
        v = df[["query_text", "target"]].copy()
        v["query_text"] = [augment(t, rng) for t in v.query_text]
        views.append(v)
    return prep(pd.concat(views, ignore_index=True))


def add_pseudo(df, name):
    ps = pd.read_csv(ROOT / f"data/{name}.csv")[["query_text", "target"]]
    return pd.concat([df[["query_text", "target"]], ps], ignore_index=True)


def main():
    use_aug = "--aug" in sys.argv
    use_ps = "--pseudo" in sys.argv
    tr = prep(pd.read_csv(ROOT / "data/train.csv"))
    te = prep(pd.read_csv(ROOT / "data/test.csv"))
    y = tr.target.values

    save_run = Path(sys.argv[sys.argv.index("--save_run") + 1]) if "--save_run" in sys.argv else None
    oof = np.zeros(len(tr), dtype=int)
    margins = np.zeros((len(tr), 56))
    folds = []
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y)):
        base = add_pseudo(tr.iloc[a], f"pseudo_f{k}_t92") if use_ps else tr.iloc[a]
        fit_df = with_aug(base, SEED + k) if use_aug else prep(base)
        Xa, (Xb, Xt) = featurize(fit_df, [tr.iloc[b], te] if save_run else [tr.iloc[b], te.iloc[:0]])
        clf = LinearSVC(C=0.5, class_weight="balanced").fit(Xa, fit_df.target.values)
        assert list(clf.classes_) == list(range(56))
        margins[b] = clf.decision_function(Xb)
        oof[b] = margins[b].argmax(1)
        folds.append((b, clf.decision_function(Xt) if save_run else None))
        print(f"fold {k}: macro-F1 {f1_score(y[b], oof[b], average='macro'):.4f}", flush=True)

    if save_run:
        def nll(t):
            z = margins * t
            z = z - z.max(1, keepdims=True)
            return float(np.mean(np.log(np.exp(z).sum(1)) - z[np.arange(len(y)), y]))
        temp = min(np.arange(1, 41, 0.5), key=nll)
        print(f"temperature {temp} (OOF NLL {nll(temp):.4f})")
        for k, (b, tm) in enumerate(folds):
            d = save_run / f"fold{k}"
            d.mkdir(parents=True, exist_ok=True)
            np.save(d / "val_idx.npy", b)
            np.save(d / "val_logits.npy", (margins[b] * temp).astype(np.float32))
            np.save(d / "test_logits.npy", (tm * temp).astype(np.float32))
            (d / "metrics.json").write_text(json.dumps({"best_score": f1_score(y[b], oof[b], average="macro"),
                                                       "temperature": temp, "aug": use_aug}))
        print("wrote fold outputs to", save_run)

    print(f"CV macro-F1 {f1_score(y, oof, average='macro'):.4f}")
    for flag, name in [(True, "Devanagari"), (False, "Roman")]:
        m = tr.is_dev.values == flag
        print(f"  {name:10s} rows {m.sum():5d}  macro-F1 {f1_score(y[m], oof[m], average='macro'):.4f}")

    base = add_pseudo(tr, "pseudo_all_t92") if use_ps else tr
    fit_df = with_aug(base, SEED) if use_aug else prep(base)
    X, (Xt,) = featurize(fit_df, [te])
    clf = LinearSVC(C=0.5, class_weight="balanced").fit(X, fit_df.target.values)
    pred = clf.predict(Xt)
    if save_run:  # the single full-data SVM used at inference (param cap: coef_ + intercept_)
        d = save_run / "full"
        d.mkdir(parents=True, exist_ok=True)
        np.save(d / "test_logits.npy", (clf.decision_function(Xt) * temp).astype(np.float32))
        n_params = int(clf.coef_.size + clf.intercept_.size)
        (d / "metrics.json").write_text(json.dumps({"n_params": n_params, "temperature": temp, "aug": use_aug}))
        print(f"full-data SVM: {n_params:,} params -> {d}")
    out = ROOT / ("submissions/baseline_tfidf_aug.csv" if use_aug else "submissions/baseline_tfidf.csv")
    pd.DataFrame({"example_id": te.example_id, "target": pred}).to_csv(out, index=False)
    print("wrote", out, "test Devanagari share", round(te.is_dev.mean(), 3))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/src/postprocess.py
"""
Blend fold outputs of one or more runs, tune the out-of-scope rule and per-class biases on OOF, write a submission.

    python src/postprocess.py runs/muril                       # one model
    python src/postprocess.py runs/muril runs/hingroberta      # blend (equal weights)
    python src/postprocess.py runs/muril runs/hingroberta --weights 0.6 0.4 --pseudo 0.92
    python src/postprocess.py runs/muril_lr6 runs/tfidf_aug --search   # grid-search blend weights on OOF
    python src/postprocess.py runs/muril_lr6 runs/tfidf_aug --search --temps --group_vote --prior_em --stack

Opt-in extras (each prints its effect; keep only what improves the test-like exam):
  --temps       fit one softmax temperature per run on OOF log-loss before blending
  --group_vote  average probabilities over queries that are the same after removing mined fillers
                (13% of test rows are filler variants of one query; batch-only, the live demo predicts one query)
  --prior_em    Saerens EM: re-estimate test class priors from the posteriors and re-weight (label shift)
  --stack       logistic-regression stacker on the runs' OOF log-probs instead of a weighted average

Each run dir holds fold{k}/{val_idx,val_logits,test_logits}.npy written by kaggle/train.py.
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from testlike import testlike_scores

ROOT = Path(__file__).resolve().parents[1]
OOS = 55


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(d, n_train):
    """Temperature minimising OOF log-loss of a run's raw logits."""
    idx, lo = [], []
    for f in sorted(p for p in Path(d).glob("fold*") if (p / "metrics.json").exists()):
        idx.append(np.load(f / "val_idx.npy")); lo.append(np.load(f / "val_logits.npy"))
    idx, lo = np.concatenate(idx), np.concatenate(lo)
    y = Y[idx]

    def nll(t):
        z = lo / t
        z = z - z.max(1, keepdims=True)
        return float(np.mean(np.log(np.exp(z).sum(1)) - z[np.arange(len(y)), y]))
    return min(np.r_[np.arange(0.3, 3.0, 0.05)], key=nll)


def load_run(d, n_train, temp=1.0):
    d = Path(d)
    folds = sorted(p for p in d.glob("fold*") if (p / "metrics.json").exists())
    if not folds:
        raise SystemExit(f"no finished folds in {d}")
    oof = np.full((n_train, 56), np.nan)
    tests = []
    for f in folds:
        idx = np.load(f / "val_idx.npy")
        oof[idx] = softmax(np.load(f / "val_logits.npy") / temp)
        tests.append(softmax(np.load(f / "test_logits.npy") / temp))
    covered = ~np.isnan(oof[:, 0])
    print(f"{d.name}: {len(folds)} folds, OOF coverage {covered.mean():.2f}, "
          f"OOF macro-F1 {f1_score(Y[covered], oof[covered].argmax(1), average='macro'):.4f}")
    return oof, np.mean(tests, 0), covered


def apply_rule(p, bias, tau):
    pred = (np.log(p + 1e-9) + bias).argmax(1)
    pred[p[:, :OOS].max(1) < tau] = OOS  # low confidence on every in-scope intent -> fallback
    return pred


def tune(p, y):
    bias = np.zeros(56)
    taus = [0.0, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
    best_tau = max(taus, key=lambda t: f1_score(y, apply_rule(p, bias, t), average="macro"))
    best = f1_score(y, apply_rule(p, bias, best_tau), average="macro")
    for _ in range(2):  # coordinate ascent on per-class log-prob offsets
        for c in range(56):
            for d in (-0.6, -0.3, 0.3, 0.6):
                b = bias.copy(); b[c] += d
                s = f1_score(y, apply_rule(p, b, best_tau), average="macro")
                if s > best + 1e-4:
                    best, bias = s, b
    return bias * 0.5, best_tau  # shrink: OOF is train-distribution, test is shifted


def group_ids(texts):
    """Group id per query: identical after clean() and removing mined filler words (groups of size 1 get -1)."""
    from augment import mine_fillers
    from normalize import clean
    pre, suf = mine_fillers()
    fill = {w for f in pre + suf for w in f.split()}
    core = pd.Series([" ".join(w for w in clean(t).split() if w not in fill) for t in texts])
    ok = core.str.split().str.len() >= 3
    size = core.map(core.value_counts())
    gid = pd.factorize(core)[0]
    gid[~(ok & (size >= 2)).values] = -1
    return gid


def group_vote(p, gid):
    p = p.copy()
    for g in np.unique(gid[gid >= 0]):
        m = gid == g
        p[m] = p[m].mean(0)
    return p


def prior_em(p, prior_train, iters=20, damp=0.5):
    """Saerens et al. (2002) EM for label shift; damped towards the train prior for stability."""
    q = prior_train.copy()
    for _ in range(iters):
        post = p * (q / prior_train)
        post /= post.sum(1, keepdims=True)
        q = damp * post.mean(0) + (1 - damp) * prior_train
    return post, q


def stack_fit(oofs, y, C=1.0):
    """Multinomial LR on concatenated log-probs; returns (5-fold OOF probs, model fitted on all rows)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    X = np.hstack([np.log(o + 1e-9) for o in oofs])
    out = np.zeros((len(y), 56))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(X, y):
        out[b] = LogisticRegression(C=C, max_iter=3000).fit(X[a], y[a]).predict_proba(X[b])
    return out, LogisticRegression(C=C, max_iter=3000).fit(X, y)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--weights", type=float, nargs="*")
    ap.add_argument("--out", default=None)
    ap.add_argument("--search", action="store_true", help="grid-search blend weights (step 0.1) on OOF macro-F1")
    ap.add_argument("--pseudo", type=float, default=0.0, help="write confident test rows (prob >= this) for round 2")
    ap.add_argument("--temps", action="store_true")
    ap.add_argument("--group_vote", action="store_true")
    ap.add_argument("--prior_em", action="store_true")
    ap.add_argument("--stack", action="store_true")
    a = ap.parse_args()
    tr = pd.read_csv(ROOT / "data/train.csv"); te = pd.read_csv(ROOT / "data/test.csv")
    global Y
    Y = tr.target.values
    w = a.weights or [1.0] * len(a.runs)
    temps = [fit_temperature(r, len(tr)) if a.temps else 1.0 for r in a.runs]
    if a.temps:
        print("temperatures", dict(zip(a.runs, np.round(temps, 2))))
    loaded = [load_run(r, len(tr), t) for r, t in zip(a.runs, temps)]
    mask = np.logical_and.reduce([c for _, _, c in loaded])
    if a.search and len(a.runs) > 1:
        import itertools
        grid = [g for g in itertools.product(range(11), repeat=len(a.runs)) if sum(g) == 10]
        def score(g):
            o = sum(gi * l[0][mask] for gi, l in zip(g, loaded))
            return f1_score(Y[mask], o.argmax(1), average="macro")
        res = sorted(((score(g), g) for g in grid), reverse=True)
        for sc, g in res[:5]:
            print(f"  weights {[x / 10 for x in g]} -> OOF raw macro-F1 {sc:.4f}")
        w = [x / 10 for x in res[0][1]]
        print(f"search picked weights {w} on {mask.sum()} OOF rows")
    oof = sum(wi * o for wi, (o, _, _) in zip(w, loaded)) / sum(w)
    test = sum(wi * t for wi, (_, t, _) in zip(w, loaded)) / sum(w)
    if a.stack:
        st_oof, st = stack_fit([o[mask] for o, _, _ in loaded], Y[mask])
        print(f"stacker: OOF raw {f1_score(Y[mask], st_oof.argmax(1), average='macro'):.4f} "
              f"vs weighted average {f1_score(Y[mask], oof[mask].argmax(1), average='macro'):.4f}")
        oof = oof.copy(); oof[mask] = st_oof
        test = st.predict_proba(np.hstack([np.log(t + 1e-9) for _, t, _ in loaded]))
    if a.group_vote:
        g_tr, g_te = group_ids(tr.query_text), group_ids(te.query_text)
        before = f1_score(Y[mask], oof[mask].argmax(1), average="macro")
        oof = oof.copy(); oof[mask] = group_vote(oof[mask], g_tr[mask])
        print(f"group vote: train groups cover {np.mean(g_tr >= 0):.3f} of rows, OOF raw {before:.4f} -> "
              f"{f1_score(Y[mask], oof[mask].argmax(1), average='macro'):.4f}; test groups cover {np.mean(g_te >= 0):.3f}, "
              f"changed {np.mean(group_vote(test, g_te).argmax(1) != test.argmax(1)):.3f} of test argmax")
        test = group_vote(test, g_te)
    if a.prior_em:
        prior = np.bincount(Y, minlength=56) / len(Y)
        new, q = prior_em(test, prior)
        top = np.argsort(-(q / prior))[:5]
        print(f"prior EM: changed {np.mean(new.argmax(1) != test.argmax(1)):.3f} of test argmax; "
              f"most up-weighted classes {dict(zip(top.tolist(), np.round(q[top] / prior[top], 2).tolist()))}")
        test = new
    p, y = oof[mask], Y[mask]
    raw = f1_score(y, p.argmax(1), average="macro")
    bias, tau = tune(p, y)
    tuned = f1_score(y, apply_rule(p, bias, tau), average="macro")
    pred = apply_rule(test, bias, tau)
    print(f"blend OOF macro-F1 raw {raw:.4f} -> tuned {tuned:.4f} (tau {tau}, shrunk bias)")
    full = np.zeros(len(Y), int)
    full[mask] = apply_rule(p, bias, tau)
    tl = testlike_scores(Y, full, mask)
    if "testlike" in tl:
        print(f"test-like exam (tuned, {tl['n_testlike']} rows): {tl['testlike']:.4f} -> estimated public LB ~{tl['est_lb']:.3f} (+-0.01)")
    print(f"test: predicted class-55 share {np.mean(pred == OOS):.3f} (train share {np.mean(Y == OOS):.3f})")
    tag = "+".join(Path(r).name for r in a.runs)
    out = Path(a.out or ROOT / f"submissions/{tag}.csv")
    out.parent.mkdir(exist_ok=True)
    pd.DataFrame({"example_id": te.example_id, "target": pred}).to_csv(out, index=False)
    np.save(out.with_suffix(".probs.npy"), test)
    (out.with_suffix(".json")).write_text(json.dumps({"runs": a.runs, "weights": w, "temps": [float(t) for t in temps],
                                                      "extras": {k: getattr(a, k) for k in ("temps", "group_vote", "prior_em", "stack")},
                                                      "oof_raw": raw, "oof_tuned": tuned,
                                                      "tau": tau, "bias": bias.round(3).tolist(),
                                                      "testlike": tl.get("testlike"), "est_lb": tl.get("est_lb")}, indent=1))
    print("wrote", out)
    if a.pseudo:
        conf = test.max(1)
        keep = conf >= a.pseudo
        ps = pd.DataFrame({"query_text": te.query_text[keep], "target": test.argmax(1)[keep], "weight": conf[keep].round(3)})
        pth = ROOT / f"data/pseudo_{tag}.csv"
        ps.to_csv(pth, index=False)
        print(f"pseudo-labels: {keep.sum()} / {len(te)} test rows (prob >= {a.pseudo}) -> {pth}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/src/testlike.py
"""
Test-like local exam (adversarial validation). Uses test TEXT only: no test labels, no hand labelling, no new sentences.

A classifier learns to tell train queries from test queries on the script-neutral phonetic key (so it looks at
wording and content, not just Devanagari vs Roman). Train rows it mistakes for test rows are the most test-like
real examples we have; their out-of-fold predictions form a local exam that should track the public LB better
than plain CV.

    python src/testlike.py            # writes data/testlike.csv (row, p_test, testlike) and prints the AUC

Other scripts call testlike_scores(y, pred, covered) to report macro-F1 on all / test-like / test-like Roman rows.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, str(Path(__file__).parent))
from normalize import phonetic_key  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "data/testlike.csv"
TOP_FRAC = 0.25  # share of train rows kept as the test-like exam
# Public LB ~= tuned test-like macro-F1 - LB_OFFSET. Real submissions so far (test-like -> public LB):
#   v002 TF-IDF raw 0.8724 -> 0.81230 (0.060)   v003 MuRIL fold0 tuned 0.8831 -> 0.81917 (0.064)
#   v005 full MuRIL + TF-IDF tuned 0.9086 -> 0.84464 (0.064)
#   v007 full HingRoBERTa + TF-IDF tuned 0.9191 -> 0.86786 (0.051)
# The gap depends on the encoder: the MuRIL pipelines agree on 0.064, HingRoBERTa transferred to the real test
# better than the proxy predicted (0.051). The proxy ranked v007 above v005 correctly; keep 0.064 as the
# conservative default and trust the ranking more than the exact number.
LB_OFFSET = 0.064


def build():
    tr = pd.read_csv(ROOT / "data/train.csv")
    te = pd.read_csv(ROOT / "data/test.csv")
    text = pd.concat([tr.query_text, te.query_text], ignore_index=True).map(phonetic_key)
    is_test = np.r_[np.zeros(len(tr), int), np.ones(len(te), int)]
    p = np.zeros(len(text))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(text, is_test):
        vecs = [TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True),
                TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)]
        Xa = hstack([v.fit_transform(text.iloc[a]) for v in vecs]).tocsr()
        Xb = hstack([v.transform(text.iloc[b]) for v in vecs]).tocsr()
        clf = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced").fit(Xa, is_test[a])
        p[b] = clf.predict_proba(Xb)[:, 1]
    auc = roc_auc_score(is_test, p)
    pt = p[: len(tr)]
    flag = pt >= np.quantile(pt, 1 - TOP_FRAC)
    dev = tr.query_text.str.contains("[ऀ-ॿ]").values
    pd.DataFrame({"row": np.arange(len(tr)), "p_test": pt.round(4), "testlike": flag.astype(int)}).to_csv(OUT, index=False)
    print(f"train-vs-test AUC {auc:.3f} (0.5 = indistinguishable)")
    print(f"test-like rows: {flag.sum()} of {len(tr)} | Roman share {(~dev[flag]).mean():.2f} "
          f"(all train {(~dev).mean():.2f}, test {(~te.query_text.str.contains('[ऀ-ॿ]')).mean():.2f})")
    print("wrote", OUT.relative_to(ROOT))


def testlike_mask(n_train):
    if not OUT.exists():
        return None
    d = pd.read_csv(OUT)
    assert len(d) == n_train, "data/testlike.csv is stale; rerun src/testlike.py"
    return d.testlike.values.astype(bool)


def testlike_scores(y, pred, covered, texts=None):
    """Macro-F1 on covered rows: all and test-like (+ Roman-script test-like rows, diagnostic only:
    that subset ranked TF-IDF above MuRIL, the opposite of the LB, so don't decide on it)."""
    m = testlike_mask(len(y))
    out = {"all": f1_score(y[covered], pred[covered], average="macro")}
    if m is not None:
        tl = covered & m
        out["testlike"] = f1_score(y[tl], pred[tl], average="macro")
        if texts is not None:
            rom = ~pd.Series(texts).str.contains("[ऀ-ॿ]").values
            out["testlike_roman"] = f1_score(y[tl & rom], pred[tl & rom], average="macro")
        out["n_testlike"] = int(tl.sum())
        out["est_lb"] = out["testlike"] - LB_OFFSET
    return out


if __name__ == "__main__":
    build()


In [ ]:
%%writefile {REPO}/kaggle/train.py
"""
Fine-tune a multilingual encoder for 56-way Hindi/Hinglish intent classification (Kaggle GPU).

Resumable: every finished fold writes its outputs to <out>/<run>/fold{k}/ and is skipped on re-run.

    python kaggle/train.py --model muril --folds 0 1 2 3 4
    python kaggle/train.py --model hingroberta --folds 0 1 2 3 4
    python kaggle/train.py --model muril --full --epochs 5          # all data, saves weights for demo

Opt-in upgrades (all off by default, so earlier runs reproduce exactly):
    --importance 0.5            up-weight test-like train rows (data/testlike.csv p_test rank), mean weight 1
    --sample_weights data/sample_weights.csv   per-row loss weights (src/noise_weights.py: suspected label noise)
    --logit_adj 1.0 --class_weight 0           logit-adjusted loss (Menon et al. 2021) instead of class weights
    --ema 0.999                 evaluate/save an exponential moving average of the weights
    --fgm 1.0                   FGM adversarial perturbation of the word embeddings (noise robustness)
    --teacher t.npz --kd_alpha 0.5 --kd_T 2 [--kd_test 1]  distil soft labels (npz: train, test prob arrays);
                                with --kd_test 1 the test queries join training as soft-label-only rows
    --init_from runs/mlm_muril  start from a domain-adapted encoder (kaggle/mlm_pretrain.py)

Outputs per fold:  val_idx.npy  val_logits.npy  test_logits.npy  test_emb.npy  metrics.json
"""
import argparse
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from augment import augment  # noqa: E402

MODELS = {
    "muril": "google/muril-base-cased",
    "hingroberta": "l3cube-pune/hing-roberta-mixed",
    "indicbert": "ai4bharat/IndicBERTv2-MLM-Sam-TLM",
    "xlmr": "xlm-roberta-base",
    "e5small": "intfloat/multilingual-e5-small",
}
N_CLASSES = 56
PARAM_CAP = 300_000_000


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="muril", help=f"alias {list(MODELS)} or any HF id")
    p.add_argument("--data", default=None, help="dir with train.csv/test.csv (auto-detected on Kaggle)")
    p.add_argument("--out", default=None, help="output root (default /kaggle/working/runs or ./runs)")
    p.add_argument("--run", default=None, help="run name (default = model alias)")
    p.add_argument("--folds", type=int, nargs="+", default=[0, 1, 2, 3, 4])
    p.add_argument("--n_folds", type=int, default=5)
    p.add_argument("--full", action="store_true", help="train on all rows, save weights")
    p.add_argument("--epochs", type=int, default=6)
    p.add_argument("--lr", type=float, default=3e-5)
    p.add_argument("--head_lr", type=float, default=5e-4)
    p.add_argument("--llrd", type=float, default=0.9, help="layer-wise lr decay")
    p.add_argument("--bs", type=int, default=32)
    p.add_argument("--max_len", type=int, default=64)
    p.add_argument("--aug_p", type=float, default=0.6, help="prob a training row is replaced by an augmented view")
    p.add_argument("--label_smoothing", type=float, default=0.05)
    p.add_argument("--pseudo", default=None, help="csv(query_text,target,weight) of pseudo-labelled test rows")
    p.add_argument("--fp16", type=int, default=1)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--limit", type=int, default=0, help="smoke test: use first N train rows")
    p.add_argument("--importance", type=float, default=0.0)
    p.add_argument("--sample_weights", default=None)
    p.add_argument("--logit_adj", type=float, default=0.0)
    p.add_argument("--class_weight", type=int, default=1)
    p.add_argument("--ema", type=float, default=0.0)
    p.add_argument("--fgm", type=float, default=0.0)
    p.add_argument("--teacher", default=None)
    p.add_argument("--kd_alpha", type=float, default=0.5)
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--kd_test", type=int, default=0)
    p.add_argument("--init_from", "--init", dest="init_from", default=None,
                   help="start from a local encoder checkpoint (kaggle/mlm_pretrain.py output); tokenizer still from --model")
    return p.parse_args()


def find_data(arg):
    for d in [arg, "/kaggle/input/anveshan-data-sprint-2-nlp", ROOT / "data"]:
        if d and (Path(d) / "train.csv").exists():
            return Path(d)
    hits = list(Path("/kaggle/input").glob("**/train.csv")) if Path("/kaggle/input").exists() else []
    if hits:
        return hits[0].parent
    raise FileNotFoundError("train.csv not found; pass --data")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


class Net(nn.Module):
    def __init__(self, name, n_msd=5):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)  # an --init_from dir works too (MLM head weights are dropped)
        self.drops = nn.ModuleList(nn.Dropout(0.1 * (i + 1) / n_msd + 0.05) for i in range(n_msd))
        self.head = nn.Linear(self.enc.config.hidden_size, N_CLASSES)

    def forward(self, ids, mask):
        h = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(h.dtype)
        emb = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        if self.training:  # multi-sample dropout
            logits = torch.stack([self.head(d(emb)) for d in self.drops]).mean(0)
        else:
            logits = self.head(emb)
        return logits, emb


class DS(torch.utils.data.Dataset):
    """label -100 = soft-label-only row (distillation on test queries)."""
    def __init__(self, texts, labels, weights, aug_p=0.0, seed=0, soft=None):
        self.t, self.y, self.w, self.aug_p, self.seed, self.epoch = list(texts), list(labels), list(weights), aug_p, seed, 0
        self.soft = soft  # (n, 56) float32 teacher probabilities or None

    def __len__(self):
        return len(self.t)

    def __getitem__(self, i):
        t = self.t[i]
        if self.aug_p:
            r = random.Random(hash((self.seed, self.epoch, i)))
            if r.random() < self.aug_p:
                t = augment(t, r)
        s = self.soft[i] if self.soft is not None else np.zeros(N_CLASSES, np.float32)
        return t, self.y[i], self.w[i], s


def loader(ds, tok, bs, max_len, shuffle):
    def collate(b):
        t, y, w, s = zip(*b)
        enc = tok(list(t), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        return (enc["input_ids"], enc["attention_mask"], torch.tensor(y), torch.tensor(w, dtype=torch.float),
                torch.tensor(np.stack(s), dtype=torch.float))
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate,
                                       num_workers=2 if torch.cuda.is_available() else 0)  # macOS spawn can't pickle collate


def param_groups(model, a):
    enc = model.enc
    layers = getattr(getattr(enc, "encoder", None), "layer", None)
    if layers is None:
        return [{"params": enc.parameters(), "lr": a.lr}, {"params": model.head.parameters(), "lr": a.head_lr}]
    n = len(layers)
    groups = [{"params": model.head.parameters(), "lr": a.head_lr}]
    for i, layer in enumerate(layers):
        groups.append({"params": layer.parameters(), "lr": a.lr * a.llrd ** (n - 1 - i)})
    layer_ids = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in enc.parameters() if id(p) not in layer_ids]  # embeddings, pooler
    groups.append({"params": rest, "lr": a.lr * a.llrd ** n})
    for g in groups:
        g["params"] = list(g["params"])
    return groups


@torch.no_grad()
def predict(model, dl, dev, fp16):
    model.eval()
    L, E = [], []
    for ids, mask, *_ in dl:
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=fp16):
            lo, em = model(ids.to(dev), mask.to(dev))
        L.append(lo.float().cpu()); E.append(em.float().cpu())
    return torch.cat(L).numpy(), torch.cat(E).numpy().astype(np.float16)


def swap_ema(model, ema):
    """Exchange live weights and EMA weights in place (call twice to restore)."""
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in ema:
                tmp = p.data.clone(); p.data.copy_(ema[n]); ema[n].copy_(tmp)


def train_one(a, name, tok, tr_t, tr_y, tr_w, val_sets, test_t, out_dir, dev, soft=None):
    """Train; pick best epoch on mean macro-F1 over val_sets (clean + shifted). Returns metrics."""
    seed_all(a.seed)
    model = Net(a.init_from or name).to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    assert n_params <= PARAM_CAP, f"{n_params} params > cap"
    hard = np.array([v for v in tr_y if v >= 0])
    counts = np.bincount(hard, minlength=N_CLASSES).astype(float)
    cw = torch.tensor((counts.clip(1) ** -0.5) / np.mean(counts.clip(1) ** -0.5), dtype=torch.float, device=dev)
    log_prior = torch.tensor(np.log(counts.clip(1) / counts.sum()), dtype=torch.float, device=dev)
    ds = DS(tr_t, tr_y, tr_w, a.aug_p, a.seed, soft)
    dl = loader(ds, tok, a.bs, a.max_len, True)
    opt = torch.optim.AdamW(param_groups(model, a), weight_decay=0.01)
    steps = a.epochs * len(dl)
    sch = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)
    use_fp16 = bool(a.fp16) and dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
    ema = {n: p.detach().clone() for n, p in model.named_parameters()} if a.ema else None
    emb_w = model.enc.get_input_embeddings().weight
    step = 0

    def loss_fn(logits, y, w, s):
        z = logits.float()
        has_hard = y >= 0
        ce = F.cross_entropy(z + a.logit_adj * log_prior if a.logit_adj else z, y,
                             weight=cw if a.class_weight else None, label_smoothing=a.label_smoothing,
                             ignore_index=-100, reduction="none")
        if soft is None:
            return (ce * w).mean()
        st = s.clamp_min(1e-8) ** (1.0 / a.kd_T)
        st = st / st.sum(1, keepdim=True)
        kd = (st * (st.log() - F.log_softmax(z / a.kd_T, 1))).sum(1) * a.kd_T ** 2
        row = torch.where(has_hard, (1 - a.kd_alpha) * ce + a.kd_alpha * kd, kd)
        return (row * w).mean()

    best, best_state, hist = -1.0, None, []
    for ep in range(a.epochs):
        ds.epoch = ep
        model.train(); t0 = time.time(); tot = 0.0
        for ids, mask, y, w, s in dl:
            ids, mask, y, w, s = ids.to(dev), mask.to(dev), y.to(dev), w.to(dev), s.to(dev)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_fp16):
                logits, _ = model(ids, mask)
            loss = loss_fn(logits, y, w, s)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if a.fgm and emb_w.grad is not None:  # FGM: one adversarial step on the embedding matrix
                g = emb_w.grad.float()
                norm = g.norm()
                if torch.isfinite(norm) and norm > 0:
                    r = (a.fgm * g / norm).to(emb_w.dtype)
                    emb_w.data.add_(r)
                    with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_fp16):
                        adv_logits, _ = model(ids, mask)
                    scaler.scale(loss_fn(adv_logits, y, w, s)).backward()
                    emb_w.data.sub_(r)
                    del r
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            step += 1
            if ema is not None:
                d = min(a.ema, (1 + step) / (10 + step))
                with torch.no_grad():
                    for n, p in model.named_parameters():
                        ema[n].mul_(d).add_(p.detach(), alpha=1 - d)
            tot += loss.item()
        rec = {"epoch": ep, "loss": round(tot / len(dl), 4), "sec": round(time.time() - t0)}
        if ema is not None:
            swap_ema(model, ema)  # evaluate (and snapshot) the EMA weights
        if val_sets:
            for vname, (vt, vy) in val_sets.items():
                lo, _ = predict(model, loader(DS(vt, vy, [1.0] * len(vy)), tok, 128, a.max_len, False), dev, use_fp16)
                rec[vname] = round(f1_score(vy, lo.argmax(1), average="macro"), 4)
            score = np.mean([rec[v] for v in val_sets])
        else:
            score = ep  # full mode: keep last epoch
        hist.append(rec); print(json.dumps(rec), flush=True)
        if score > best:
            best = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ema is not None:
            swap_ema(model, ema)  # back to live weights for the next epoch
    model.load_state_dict(best_state)
    res = {"best_score": float(best), "best_epoch": int(max(hist, key=lambda r: np.mean([r[v] for v in val_sets]) if val_sets else r["epoch"])["epoch"]),
           "history": hist, "n_params": int(n_params)}
    tdl = loader(DS(test_t, [0] * len(test_t), [1.0] * len(test_t)), tok, 128, a.max_len, False)
    tl, te = predict(model, tdl, dev, use_fp16)
    np.save(out_dir / "test_logits.npy", tl); np.save(out_dir / "test_emb.npy", te)
    if val_sets:
        vt, vy = val_sets["val_clean"]
        vl, ve = predict(model, loader(DS(vt, vy, [1.0] * len(vy)), tok, 128, a.max_len, False), dev, use_fp16)
        np.save(out_dir / "val_logits.npy", vl); np.save(out_dir / "val_emb.npy", ve)
    return model, res


def main():
    a = parse()
    name = MODELS.get(a.model, a.model)
    run = a.run or a.model
    out_root = Path(a.out or ("/kaggle/working/runs" if Path("/kaggle/working").exists() else ROOT / "runs")) / run
    out_root.mkdir(parents=True, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = find_data(a.data)
    tr = pd.read_csv(data / "train.csv"); te = pd.read_csv(data / "test.csv")
    if a.limit:
        tr = tr.sample(a.limit, random_state=a.seed).reset_index(drop=True); te = te.head(a.limit)
    try:
        commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        commit = os.environ.get("GIT_COMMIT", "unknown")  # code shipped as an archive (Kaggle CLI runner)
    cfg = {**vars(a), "hf_name": name, "commit": commit, "device": str(dev), "data": str(data)}
    (out_root / "config.json").write_text(json.dumps(cfg, indent=2))
    print(json.dumps(cfg), flush=True)
    tok = AutoTokenizer.from_pretrained(name)

    w_tr = np.ones(len(tr))
    if a.sample_weights:
        sw = pd.read_csv(a.sample_weights)
        assert len(sw) == len(tr) or a.limit, "sample_weights rows != train rows"
        if not a.limit:
            w_tr *= sw.weight.values
        print(f"sample weights: {np.sum(w_tr < 1)} down-weighted rows", flush=True)
    if a.importance:
        tl = pd.read_csv(ROOT / "data/testlike.csv")
        if not a.limit:
            rank = tl.p_test.rank(pct=True).values
            w_tr *= (1 - a.importance) + a.importance * 2 * rank
        print(f"importance weighting alpha={a.importance}: weight range {w_tr.min():.2f}-{w_tr.max():.2f}", flush=True)
    soft_tr = soft_te = None
    if a.teacher:
        z = np.load(a.teacher)
        soft_tr, soft_te = z["train"].astype(np.float32), z["test"].astype(np.float32)
        if a.limit:
            soft_tr, soft_te = soft_tr[: len(tr)], soft_te[: len(te)]
        assert soft_tr.shape == (len(tr), N_CLASSES) and soft_te.shape == (len(te), N_CLASSES)
        print(f"teacher: {a.teacher} kd_alpha={a.kd_alpha} T={a.kd_T} test rows as soft-only: {bool(a.kd_test)}", flush=True)

    extra_t, extra_y, extra_w, extra_s = [], [], [], []
    if a.pseudo:
        ps = pd.read_csv(a.pseudo)
        extra_t, extra_y, extra_w = ps.query_text.tolist(), ps.target.astype(int).tolist(), ps.weight.astype(float).tolist()
        extra_s = [np.eye(N_CLASSES, dtype=np.float32)[v] for v in extra_y]
        print(f"pseudo rows: {len(ps)}", flush=True)
    if a.teacher and a.kd_test:
        extra_t += te.query_text.tolist(); extra_y += [-100] * len(te); extra_w += [1.0] * len(te)
        extra_s += list(soft_te)

    def soft_for(idx):
        return None if soft_tr is None else np.concatenate([soft_tr[idx], np.array(extra_s, np.float32).reshape(-1, N_CLASSES)])

    y = tr.target.values
    if a.full:
        fdir = out_root / "full"; fdir.mkdir(exist_ok=True)
        if (fdir / "metrics.json").exists():
            print("full run already done; delete", fdir, "to redo"); return
        model, res = train_one(a, name, tok, tr.query_text.tolist() + extra_t, list(y) + extra_y,
                               list(w_tr) + extra_w, None, te.query_text.tolist(), fdir, dev, soft_for(np.arange(len(tr))))
        torch.save(model.state_dict(), fdir / "model.pt")
        tok.save_pretrained(fdir / "tokenizer")
        (fdir / "metrics.json").write_text(json.dumps(res, indent=2))
        return

    skf = list(StratifiedKFold(a.n_folds, shuffle=True, random_state=42).split(tr, y))
    for k in a.folds:
        fdir = out_root / f"fold{k}"
        if (fdir / "metrics.json").exists():
            print(f"fold {k} done, skipping", flush=True); continue
        fdir.mkdir(exist_ok=True)
        tri, vai = skf[k]
        vt = tr.query_text.iloc[vai].tolist(); vy = y[vai].tolist()
        rs = random.Random(1000 + k)
        val_sets = {"val_clean": (vt, vy), "val_shift": ([augment(t, rs) for t in vt], vy)}
        print(f"=== fold {k}: train {len(tri)} val {len(vai)}", flush=True)
        _, res = train_one(a, name, tok, tr.query_text.iloc[tri].tolist() + extra_t, list(y[tri]) + extra_y,
                           list(w_tr[tri]) + extra_w, val_sets, te.query_text.tolist(), fdir, dev, soft_for(tri))
        np.save(fdir / "val_idx.npy", vai)
        (fdir / "metrics.json").write_text(json.dumps(res, indent=2))  # written last = fold complete
        print(f"fold {k} best {res['best_score']:.4f} @ epoch {res['best_epoch']}", flush=True)
        if dev.type == "cuda":
            torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


## 4) Frozen decision rule (tuned on 5-fold out-of-fold predictions of train.csv, geometric blend)
Weights [0.5, 0.5] on log-probabilities, class-55 threshold tau = 0.0, 56 per-class log-probability biases. On the 16,936 OOF rows this rule gives macro-F1 0.91853 (0.92288 on the 4,234 most test-like rows); v007: 0.91689 / 0.91915.

In [ ]:
RULE = {"weights": [0.5, 0.5], "tau": 0.0, "bias": [0.0, 0.15, 0.0, 0.0, 0.0, 0.15, -0.45, -0.45, 0.15, -0.15, 0.0, 0.0, 0.45, 0.0, 0.0, 0.3, -0.6, -0.3, -0.3, -0.3, -0.15, 0.15, 0.0, -0.3, 0.0, -0.3, -0.3, -0.3, -0.6, 0.0, -0.3, 0.45, 0.0, 0.0, -0.3, 0.0, 0.0, 0.15, -0.3, 0.0, 0.0, 0.0, 0.15, 0.0, 0.0, 0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.15], "blend": "geometric", "oof_tuned_macro_f1": 0.91853, "testlike_macro_f1": 0.92288}
json.dump(RULE, open(f'{REPO}/rule_candG.json', 'w'))
print('rule saved')

## 5) TF-IDF + LinearSVC
5-fold CV (prints macro-F1 ≈ 0.886), fits the softmax temperature on OOF log-loss, then fits one model on all augmented training rows and saves its test scores.

In [ ]:
!cd {REPO} && python src/baseline.py --aug --save_run runs/tfidf_aug

## 6) HingRoBERTa-mixed, fine-tuned on all training rows
Mean-pooled encoder + linear head, multi-sample dropout, sqrt-inverse class weights, label smoothing 0.05, layer-wise LR decay 0.95, cosine schedule, fp16, 10 epochs, lr 5e-5, 60% of rows replaced by a rule-based augmented view each epoch. Saves `model.pt` and its test scores.

In [ ]:
extra = '--limit 300 --epochs 1' if SMOKE else '--epochs 10'
!cd {REPO} && python kaggle/train.py --model hingroberta --full --lr 5e-5 --llrd 0.95 {extra} --seed {SEED} --run hing_full --out runs

## 7) Optional: re-derive the decision rule from 5-fold OOF (`RECOMPUTE_RULE = True`)
Trains HingRoBERTa on each of 5 folds (~13 min each), builds the test-like subset, and tunes tau + per-class biases on the OOF blend with `postprocess.py`. The result should match section 4.

In [ ]:
if RECOMPUTE_RULE:
    !cd {REPO} && python src/testlike.py
    !cd {REPO} && python kaggle/train.py --model hingroberta --lr 5e-5 --llrd 0.95 --epochs 10 --run hing_lr5 --out runs
    !cd {REPO} && python src/postprocess.py runs/hing_lr5 runs/tfidf_aug --weights 0.5 0.5 --out submissions/oof_rule.csv
    RULE = {**RULE, **{k: v for k, v in json.load(open(f'{REPO}/submissions/oof_rule.json')).items() if k in ('tau', 'bias', 'weights')}}
    print('re-derived tau', RULE['tau'])
else:
    print('using the frozen rule from section 4')

## 8) Blend and write `submission.csv`

In [ ]:
import numpy as np
def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
h = np.load(f'{REPO}/runs/hing_full/full/test_logits.npy')
t = np.load(f'{REPO}/runs/tfidf_aug/full/test_logits.npy')
w = np.array(RULE['weights']) / sum(RULE['weights'])
p = softmax(w[0] * np.log(softmax(h) + 1e-9) + w[1] * np.log(softmax(t) + 1e-9))   # geometric mean
pred = (np.log(p + 1e-9) + np.array(RULE['bias'])).argmax(1)
pred[p[:, :55].max(1) < RULE['tau']] = 55   # no supported intent confident enough -> out of scope
sub = pd.DataFrame({'example_id': te.example_id, 'target': pred})
sub.to_csv('/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv', index=False)
print(sub.shape, '| class-55 share', round((pred == 55).mean(), 3)); sub.head()

## 9) Parameter check (organisers' assertion)

In [ ]:
import sys; sys.path.insert(0, f'{REPO}/kaggle'); sys.path.insert(0, f'{REPO}/src')
import importlib, train as T
cfg = json.load(open(f'{REPO}/runs/hing_full/config.json'))
net = T.Net(cfg['hf_name']); net.load_state_dict(torch.load(f'{REPO}/runs/hing_full/full/model.pt', map_location='cpu'))
models = [net]
svm_params = json.load(open(f'{REPO}/runs/tfidf_aug/full/metrics.json'))['n_params']
torch_params = sum(p.numel() for m in models for p in m.parameters())
total = torch_params + svm_params
print(f'HingRoBERTa {torch_params:,} + TF-IDF/LinearSVC {svm_params:,} = {total:,}')
assert total <= 300_000_000

## 10) Agreement with the candidate file we produced (NEW-G)
GPU training is not bit-exact across machines, so expect a high but not perfect match.

In [ ]:
import gzip, base64, io
V007 = '...'
ref = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(V007))))
assert (ref.example_id.values == sub.example_id.values).all()
print(f'identical predictions: {np.mean(ref.target.values == sub.target.values):.2%} of {len(sub)} test rows')